# Progress Review 1 — Validate Streamed Storage

This notebook validates that Kafka-streamed weather events were written into MinIO.

It checks:

1. JSONL streamed event files in the raw bucket.
2. Row counts from stored streamed files.
3. Optional Delta Lake table in the lakehouse bucket.

In [1]:
import os
import json
from io import BytesIO

import boto3
import pandas as pd

In [2]:
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT_INTERNAL")
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

RAW_BUCKET = os.getenv("RAW_BUCKET")
LAKEHOUSE_BUCKET = os.getenv("LAKEHOUSE_BUCKET")

required = {
    "MINIO_ENDPOINT_INTERNAL": MINIO_ENDPOINT,
    "MINIO_ROOT_USER": MINIO_ACCESS_KEY,
    "MINIO_ROOT_PASSWORD": MINIO_SECRET_KEY,
    "RAW_BUCKET": RAW_BUCKET,
    "LAKEHOUSE_BUCKET": LAKEHOUSE_BUCKET,
}

missing = [k for k, v in required.items() if not v]
if missing:
    raise RuntimeError(f"Missing environment variables: {missing}")

print("MinIO:", MINIO_ENDPOINT)
print("Raw bucket:", RAW_BUCKET)
print("Lakehouse bucket:", LAKEHOUSE_BUCKET)

MinIO: http://minio:9000
Raw bucket: raw
Lakehouse bucket: lakehouse


In [3]:
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name=AWS_REGION,
)

print([b["Name"] for b in s3.list_buckets()["Buckets"]])

['lakehouse', 'raw', 'warehouse']


In [4]:
resp = s3.list_objects_v2(
    Bucket=RAW_BUCKET,
    Prefix="kafka_weather_events/",
)

objects = resp.get("Contents", [])

print("Streamed object count:", len(objects))

for obj in objects[-20:]:
    print(obj["Key"], obj["Size"])

Streamed object count: 8
kafka_weather_events/run_id=20260424T210732Z-8aeb0b2e/year=2026/month=04/day=24/part-00001.jsonl 6289
kafka_weather_events/run_id=20260424T210732Z-8aeb0b2e/year=2026/month=04/day=24/part-00002.jsonl 6267
kafka_weather_events/run_id=20260424T210732Z-8aeb0b2e/year=2026/month=04/day=24/part-00003.jsonl 6303
kafka_weather_events/run_id=20260424T210732Z-8aeb0b2e/year=2026/month=04/day=24/part-00004.jsonl 6227
kafka_weather_events/run_id=20260424T210732Z-8aeb0b2e/year=2026/month=04/day=24/part-00005.jsonl 6224
kafka_weather_events/run_id=20260424T210732Z-8aeb0b2e/year=2026/month=04/day=24/part-00006.jsonl 6033
kafka_weather_events/run_id=20260424T210732Z-8aeb0b2e/year=2026/month=04/day=24/part-00007.jsonl 6317
kafka_weather_events/run_id=20260424T210732Z-8aeb0b2e/year=2026/month=04/day=24/part-00008.jsonl 1257


In [5]:
all_rows = []

for obj in objects:
    key = obj["Key"]

    body = (
        s3.get_object(Bucket=RAW_BUCKET, Key=key)["Body"]
        .read()
        .decode("utf-8")
    )

    for line in body.strip().splitlines():
        if line.strip():
            row = json.loads(line)
            row["_source_object"] = key
            all_rows.append(row)

streamed_df = pd.DataFrame(all_rows)

print("Total streamed rows read from MinIO:", len(streamed_df))
streamed_df.head()

Total streamed rows read from MinIO: 72


,event_id,airport,timestamp_utc,latitude,longitude,temperature_k,temperature_c,wind_u_ms,wind_v_ms,wind_speed_ms,wind_speed_kts,precipitation_m,precipitation_mm,surface_pressure_pa,cape_j_kg,_kafka_topic,_kafka_partition,_kafka_offset,_ingested_at_utc,_source_object
0,JFK_2022-01-01T00:00:00,JFK,2022-01-01 00:00:00,40.75,286.25,282.574158,9.424164,-0.005779,1.871856,1.871865,3.638606,5.761161e-06,0.005761,101271.765625,12.700439,weather.raw,0,1,2026-04-24T21:08:55.416795+00:00,kafka_weather_events/run_id=20260424T210732Z-8...
1,JFK_2022-01-01T01:00:00,JFK,2022-01-01 01:00:00,40.75,286.25,282.615082,9.465088,0.255980,2.030057,2.046132,3.977354,1.056492e-05,0.010565,101288.007812,21.743164,weather.raw,0,2,2026-04-24T21:08:55.458213+00:00,kafka_weather_events/run_id=20260424T210732Z-8...
2,JFK_2022-01-01T02:00:00,JFK,2022-01-01 02:00:00,40.75,286.25,282.552124,9.402130,-0.076426,2.341874,2.343121,4.554652,1.248531e-05,0.012485,101208.484375,23.470459,weather.raw,0,3,2026-04-24T21:08:55.565702+00:00,kafka_weather_events/run_id=20260424T210732Z-8...
3,JFK_2022-01-01T03:00:00,JFK,2022-01-01 03:00:00,40.75,286.25,282.542694,9.392700,0.014148,2.136289,2.136336,4.152695,1.439825e-06,0.001440,101199.937500,17.780762,weather.raw,0,4,2026-04-24T21:08:55.668698+00:00,kafka_weather_events/run_id=20260424T210732Z-8...
4,JFK_2022-01-01T04:00:00,JFK,2022-01-01 04:00:00,40.75,286.25,282.262512,9.112518,0.130988,2.419828,2.423371,4.710645,9.592623e-07,0.000959,101201.640625,14.834229,weather.raw,0,5,2026-04-24T21:08:55.771407+00:00,kafka_weather_events/run_id=20260424T210732Z-8...


In [6]:
expected_columns = [
    "event_id",
    "airport",
    "timestamp_utc",
    "temperature_c",
    "wind_speed_kts",
    "precipitation_mm",
    "surface_pressure_pa",
    "cape_j_kg",
    "_kafka_topic",
    "_kafka_partition",
    "_kafka_offset",
    "_ingested_at_utc",
]

missing = [col for col in expected_columns if col not in streamed_df.columns]

if missing:
    print("Missing expected columns:", missing)
else:
    print("All expected columns are present.")

print("Rows:", len(streamed_df))
print("Airports:", streamed_df["airport"].unique())
print("Min timestamp:", streamed_df["timestamp_utc"].min())
print("Max timestamp:", streamed_df["timestamp_utc"].max())

All expected columns are present.
Rows: 72
Airports: ['JFK']
Min timestamp: 2022-01-01 00:00:00
Max timestamp: 2022-01-03 23:00:00


In [7]:
summary = {
    "total_rows": len(streamed_df),
    "unique_airports": streamed_df["airport"].nunique(),
    "airport_list": ", ".join(sorted(streamed_df["airport"].dropna().unique())),
    "min_timestamp": streamed_df["timestamp_utc"].min(),
    "max_timestamp": streamed_df["timestamp_utc"].max(),
    "min_kafka_offset": streamed_df["_kafka_offset"].min(),
    "max_kafka_offset": streamed_df["_kafka_offset"].max(),
    "stored_objects": len(objects),
}

summary_df = pd.DataFrame([summary])
summary_df

,total_rows,unique_airports,airport_list,min_timestamp,max_timestamp,min_kafka_offset,max_kafka_offset,stored_objects
0,72,1,JFK,2022-01-01 00:00:00,2022-01-03 23:00:00,1,72,8


In [8]:
try:
    from deltalake import DeltaTable

    storage_options = {
        "AWS_ENDPOINT_URL": MINIO_ENDPOINT,
        "AWS_ACCESS_KEY_ID": MINIO_ACCESS_KEY,
        "AWS_SECRET_ACCESS_KEY": MINIO_SECRET_KEY,
        "AWS_REGION": AWS_REGION,
        "AWS_ALLOW_HTTP": "true",
        "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    }

    delta_uri = f"s3://{LAKEHOUSE_BUCKET}/bronze/weather_events_delta"

    dt = DeltaTable(
        delta_uri,
        storage_options=storage_options,
    )

    delta_df = dt.to_pandas()

    print("Delta table found:", delta_uri)
    print("Delta rows:", len(delta_df))
    display(delta_df.head())

except Exception as exc:
    print("Delta table validation failed.")
    print("This is acceptable for the raw ingestion demo if JSONL storage worked.")
    print("Error:", repr(exc))

Delta table found: s3://lakehouse/bronze/weather_events_delta
Delta rows: 72


,event_id,airport,timestamp_utc,latitude,longitude,temperature_k,temperature_c,wind_u_ms,wind_v_ms,wind_speed_ms,wind_speed_kts,precipitation_m,precipitation_mm,surface_pressure_pa,cape_j_kg,_kafka_topic,_kafka_partition,_kafka_offset,_ingested_at_utc
0,JFK_2022-01-03T22:00:00,JFK,2022-01-03 22:00:00,40.75,286.25,271.122070,-2.027924,0.583468,-7.228751,7.252260,14.097234,0.000171,0.170683,101656.976562,0.000244,weather.raw,0,71,2026-04-24T21:09:02.615531+00:00
1,JFK_2022-01-03T23:00:00,JFK,2022-01-03 23:00:00,40.75,286.25,270.978241,-2.171753,0.805318,-6.382805,6.433408,12.505515,0.000056,0.055572,101771.343750,0.000244,weather.raw,0,72,2026-04-24T21:09:02.718379+00:00
2,JFK_2022-01-03T12:00:00,JFK,2022-01-03 12:00:00,40.75,286.25,273.304626,0.154633,-0.526485,-5.885190,5.908692,11.485553,0.000027,0.026651,101741.015625,0.000244,weather.raw,0,61,2026-04-24T21:09:01.575321+00:00
3,JFK_2022-01-03T13:00:00,JFK,2022-01-03 13:00:00,40.75,286.25,273.026337,-0.123657,-0.767291,-5.956977,6.006189,11.675072,0.000088,0.087893,101788.671875,0.000244,weather.raw,0,62,2026-04-24T21:09:01.678080+00:00
4,JFK_2022-01-03T14:00:00,JFK,2022-01-03 14:00:00,40.75,286.25,272.568237,-0.581757,-1.486198,-6.635692,6.800087,13.218282,0.000245,0.244968,101797.335938,0.000244,weather.raw,0,63,2026-04-24T21:09:01.784085+00:00


In [9]:
print("Progress Review 1 validation complete.")
print()
print("Evidence:")
print("1. Real ARCO-ERA5 data was downloaded.")
print("2. Weather events were replayed into Kafka topic weather.raw.")
print("3. Kafka consumer received the events.")
print("4. Streamed events were written to MinIO raw storage.")
print("5. Stored events were read back and validated from MinIO.")

Progress Review 1 validation complete.

Evidence:
1. Real ARCO-ERA5 data was downloaded.
2. Weather events were replayed into Kafka topic weather.raw.
3. Kafka consumer received the events.
4. Streamed events were written to MinIO raw storage.
5. Stored events were read back and validated from MinIO.
